# Apply embedding-based sense classifiers to the full datasets

Run from `scripts/`, after the full embedding export in `4_contextual-embeddings.ipynb`.
Input filenames match that export, including the spelling `embeddigings_full`.
The saved artifacts come from `5_embeddings-classifier.ipynb`: PhilBERTa layer 11
and mDeBERTa layer 9, concatenated in that order.

This notebook performs inference only. It loads the fitted scaler/MLP pipelines,
uses their stored input specification and training-time sense map, and processes
Parquet batches to avoid loading the full embedding datasets into memory. All
original columns and row order are preserved. Predictions are added in separate
columns rather than replacing existing LLM annotations. Output files become
available only once the entire term has been processed successfully.


In [1]:
from collections import Counter
from pathlib import Path
import os
import tempfile

import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

DATA_DIR = Path("../data")
FULL_DATA_DIR = DATA_DIR / "large_files"
CASE_STUDIES_TERMS = ["pathos", "passio"]
BATCH_SIZE = 1024  # Lower this if memory is limited.

SOURCES = {
    term: FULL_DATA_DIR / f"{term}_embeddigings_full.parquet"
    for term in CASE_STUDIES_TERMS
}
MODEL_PATHS = {
    term: DATA_DIR / f"{term}_wsd_philberta_l11_mdeberta_l9_concat.joblib"
    for term in CASE_STUDIES_TERMS
}
OUTPUTS = {
    term: FULL_DATA_DIR / f"{term}_embeddigings_full_classified.parquet"
    for term in CASE_STUDIES_TERMS
}


## Load saved classifiers and check inputs

In [2]:
# Check every input before beginning inference.
for term in CASE_STUDIES_TERMS:
    for path in (SOURCES[term], MODEL_PATHS[term]):
        if not path.is_file():
            raise FileNotFoundError(f"Missing {path}. Generate the full embeddings / saved models first.")

artifacts = {term: joblib.load(MODEL_PATHS[term]) for term in CASE_STUDIES_TERMS}
for term, artifact in artifacts.items():
    print(f"{term}: {artifact['embedding_cols']} ({artifact['merge_method']})")
    print(f"  Training labels: {artifact['label_column']}")
    print(f"  Saved CV accuracy={artifact['cv_accuracy']:.4f}, "
          f"macro-F1={artifact['cv_f1_macro']:.4f}")


pathos: ['emb_philberta_l11', 'emb_mdeberta_l9'] (concat)
  Training labels: class_qwen3.8:27b
  Saved CV accuracy=0.6970, macro-F1=0.6853
passio: ['emb_philberta_l11', 'emb_mdeberta_l9'] (concat)
  Training labels: class_qwen3.8:27b
  Saved CV accuracy=0.8100, macro-F1=0.7568


## Batched inference

In [3]:
def classify_full_parquet(source, destination, artifact, batch_size=1024):
    """Apply a saved concatenation classifier while preserving all source data."""
    source, destination = Path(source), Path(destination)
    if source.resolve() == destination.resolve():
        raise ValueError("Source and destination must be different files.")
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    if artifact["merge_method"] != "concat":
        raise ValueError("This notebook expects a saved concatenation classifier.")

    parquet = pq.ParquetFile(source)
    columns = artifact["embedding_cols"]
    missing = set(columns) - set(parquet.schema_arrow.names)
    if missing:
        raise ValueError(f"{source.name}: missing embedding columns {sorted(missing)}")
    if parquet.metadata.num_rows == 0:
        raise ValueError(f"{source.name}: input is empty.")

    prediction_col = f"{artifact['label_column']}_embedding_pred"
    sense_col = f"{prediction_col}_sense"
    if {prediction_col, sense_col} & set(parquet.schema_arrow.names):
        raise ValueError("Input already contains the output prediction columns.")

    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destination.parent,
                                     prefix=destination.name + ".", suffix=".tmp",
                                     delete=False) as handle:
        temporary = Path(handle.name)
    writer = None
    counts = Counter()
    processed = 0
    try:
        for batch_number, batch in enumerate(parquet.iter_batches(batch_size=batch_size), 1):
            table = pa.Table.from_batches([batch])
            matrices = [np.stack(table[col].to_pylist()) for col in columns]
            X = np.concatenate(matrices, axis=1)
            expected = artifact["model"].n_features_in_
            if X.ndim != 2 or X.shape[1] != expected or not np.isfinite(X).all():
                raise ValueError(f"Invalid embedding batch: expected {expected} finite features.")

            predictions = artifact["model"].predict(X)
            unknown = set(predictions) - set(artifact["label_map"])
            if unknown:
                raise ValueError(f"Predicted labels absent from the saved sense inventory: {unknown}")
            names = [artifact["label_map"][label] for label in predictions]
            table = table.append_column(prediction_col, pa.array(predictions))
            table = table.append_column(sense_col, pa.array(names, type=pa.string()))
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression="snappy")
            writer.write_table(table)
            counts.update(predictions.tolist())
            processed += len(predictions)
            if batch_number % 10 == 0:
                print(f"{source.stem}: {processed:,}/{parquet.metadata.num_rows:,} rows")
        writer.close()
        writer = None
        if processed != parquet.metadata.num_rows:
            raise RuntimeError("Output row count does not match input.")
        os.replace(temporary, destination)
    finally:
        if writer is not None:
            writer.close()
        temporary.unlink(missing_ok=True)

    print(f"Saved {destination}: {processed:,} rows")
    return {"rows": processed, "prediction_column": prediction_col,
            "sense_column": sense_col, "counts": counts}


In [4]:
applied_results = {}
for term in CASE_STUDIES_TERMS:
    applied_results[term] = classify_full_parquet(
        SOURCES[term], OUTPUTS[term], artifacts[term], batch_size=BATCH_SIZE,
    )


pathos_embeddigings_full: 10,240/16,280 rows
Saved ../data/large_files/pathos_embeddigings_full_classified.parquet: 16,280 rows
passio_embeddigings_full: 10,240/100,361 rows
passio_embeddigings_full: 20,480/100,361 rows
passio_embeddigings_full: 30,720/100,361 rows
passio_embeddigings_full: 40,960/100,361 rows
passio_embeddigings_full: 51,200/100,361 rows
passio_embeddigings_full: 61,440/100,361 rows
passio_embeddigings_full: 71,680/100,361 rows
passio_embeddigings_full: 81,920/100,361 rows
passio_embeddigings_full: 92,160/100,361 rows
Saved ../data/large_files/passio_embeddigings_full_classified.parquet: 100,361 rows


## Predicted sense distributions

In [5]:
# Distribution of predicted senses in the full datasets.
summary_rows = []
for term, result in applied_results.items():
    for label, name in artifacts[term]["label_map"].items():
        count = result["counts"].get(label, 0)
        summary_rows.append({
            "lexeme": term, "sense_id": label, "sense": name,
            "count": count, "proportion": count / result["rows"],
        })
predicted_sense_summary = pd.DataFrame(summary_rows)
predicted_sense_summary


,lexeme,sense_id,sense,count,proportion
0,pathos,0,affect_emotion,3296,0.202457
1,pathos,1,bodily_affection,3173,0.194902
2,pathos,2,property_state_modification,5987,0.367752
3,pathos,3,suffering_undergoing_event,3824,0.234889
4,passio,0,suffering_undergoing,8523,0.084923
5,passio,1,disease_pathological_condition,7546,0.075189
6,passio,2,property_accident,20541,0.204671
7,passio,3,christs_passion,42319,0.421668
8,passio,4,martyrdom_narrative,3022,0.030111
9,passio,5,affect_emotion,18410,0.183438


## Read the classified data elsewhere

The output filenames are `<term>_embeddigings_full_classified.parquet` in
`data/large_files/`. The `class_<annotation-model>_embedding_pred` column holds
predicted sense IDs; its `_sense` companion contains the names stored with the
trained classifier. Use the prediction column explicitly in downstream analyses.
The original embedding and metadata columns remain available.
